# Import Required Libraries and Create Clean Datasets

## Import Required Libraries

In [181]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib
import urllib.request
from bs4 import BeautifulSoup
import sqlite3
import sys
!{sys.executable} -m pip install "kagglehub[pandas-datasets]"
import kagglehub
# print("kagglehub works!")
from kagglehub import KaggleDatasetAdapter

## Webscrape the Hosts for Each Year

### Fetch and Locate Hosts Data

In [182]:
# Fetch HTML from Wikipedia page
hosts_url = "https://en.wikipedia.org/wiki/FIFA_World_Cup"

# Create request with custom User Agent to avoid having the request blocked
req = urllib.request.Request(hosts_url, headers={"User-Agent" : "Magic Browser"})

# Retrieve page content
con = urllib.request.urlopen(req)

# Parse the HTML using BeautifulSoup
soup = BeautifulSoup(con.read(), "html.parser")

# Find all tables with the class "wikitable"
wiki_tables = soup.find_all("table", class_=lambda x: x and "wikitable" in x)

# Print number of tables found (commented out after observing)
# print(len(wiki_tables))

# Find table with host countries and year
host_table = None
for table in wiki_tables:
    table_header = table.find_all("th") # Get all header cells
    table_header_txt = " ".join([h.text.strip().lower()
                                 for h in table_header]) # Make a list of lowercase header cells so key words can be searched for
    # Print all header strings so we can choose the most relevant table (commented out after observing)
    # print(table_header_txt)
    if "year" in table_header_txt and "host" in table_header_txt:
        host_table = table # Select the table that has "year" and "host" as headers
        break

# Print to confirm we have the right wikitable (commented out after observing)        
# print(host_table)


### Extract Hosts Data to a DataFrame and Export Raw Data to CSV

In [183]:
# Get every row from the table
rows = host_table.find_all("tr")

data = []

# Skip the first two rows (column names) and extract the rest
for row in rows[2:]:
    cols = [c.text.strip() for c 
            in row.find_all(["td", "th"])] # Extract <td> and <th> without spaces
    data.append(cols)

# Convert to DataFrame
host_df = pd.DataFrame(data, columns = ["Edition", "Year", "Host", "First Place", "Score", 
                                   "Runner-up", "Third Place", "Score2", "Fourth Place", "Nr. of Teams"])

# Print out to check DataFrame (commented out after observing)
# print(host_df)

# Save raw DataFrame as a CSV file
host_df.to_csv("hosts_and_years_raw.csv")


## Cleaning Hosts Data

### Check Data Type for Each Column

In [184]:
# Check data type for each column  (commented out after observing)
# host_df.info()

### Check for Missing Data in the Dataset

In [185]:
# Check for missing data in the dataset  (commented out after observing)
data_missing = host_df.isnull()
# data_missing

In [186]:
# Find missing data count per column (commented out after observing)
# host_df.isnull().sum()

### Drop Rows with Missing Data

##### In 1942 and 1946 no World Cup took place due to WWII
##### The 2026, 2030 and 2034 World Cups haven't taken place yet
##### There is no data for these rows so they should be dropped

#### Clean Missing Score Columns Using SQL

In [187]:
# Create a copy of the original DataFrame to prevent changing raw data
host_df1 = host_df.copy()

# Set up in-memory SQL database to allow SQL cleaning
conn = sqlite3.connect(":memory:")

# Load DataFrame into SQLite as a table
host_df1.to_sql("host_df1", conn, if_exists = "replace", index=False)

# Replace "None", "NaN" and NULL values with an empty string
host_df1 = conn.execute("""
    UPDATE host_df1
    SET Score = ""
    WHERE Score IS NULL OR Score = 'None' OR Score = 'NaN'
""")
conn.commit()

# Print the Score and Year column to check this worked
score_updated = pd.read_sql("""
    SELECT Year, Score
    FROM host_df1
""", conn)

# Print to check this worked (commented out after observing)
# print(score_updated.to_string(index = False))


#### Check we are Dropping the Correct Rows Using SQL

In [188]:
# Check we are dropping the correct rows
check_empty = pd.read_sql("""
    SELECT *
    FROM host_df1
    WHERE Score = ""
""", conn)

# Print to check (commented out after observing)
# print(check_empty.to_string(index = False))

#### Delete the Rows Using SQL

In [189]:
# Dropping the missing data
host_df1 = conn.execute("""
    DELETE FROM host_df1
    WHERE Score = ""
""")
conn.commit()

# Check this worked
rows_not_deleted = pd.read_sql("""
    SELECT *
    FROM host_df1
""", conn)

# Print to check (commented out after observing)
# print(rows_not_deleted.to_string(index = False))

#### Extract Relevant Columns Only Using SQL

In [190]:
# Extract "Year" and "Host" from the hosts_table
host_df1 = pd.read_sql("""
    SELECT Year, Host
    FROM host_df1
""", conn)

# Print the filtered DataFrame to check this worked (commented out after observing)
# print(host_df1.to_string(index=False))

## Download Elo Ratings Dataset into Pandas

##### Please refer to the README file regarding this code
##### The only complete source of Elo Ratings is a JavaScript webpage so I was not able to webscrape as I am not familiar enough with Selenium
##### Instead, this Dataset is downloaded from Kagglehub into Pandas

### Download and Locally Save the Elo Dataset

In [191]:
# Specify the file within the kagglehub dataset
target_file_path = "eloratings.csv"

# Download the specified dataset directly into a Pandas DataFrame
elo_ranking_raw = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "saifalnimri/international-football-elo-ratings",
  target_file_path,
)

# Print to check this has worked properly (commented out after observing)
# print(elo_ranking_raw)

# Save as CSV file
df.to_csv("elo_ranking_raw.csv")

/var/folders/kd/p53w9pwd7rv29w5h_tss9l580000gn/T/ipykernel_60895/3128765576.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  elo_ranking_raw = kagglehub.load_dataset(


## Cleaning Elo Data

### Check Data Type for Each Column

In [192]:
# Check data type for each column  (commented out after observing)
# elo_ranking_raw.info()

### Check for Missing Data in the Dataset

In [193]:
# Check for missing data in the dataset  (commented out after observing)
data_missing = elo_ranking_raw.isnull()
# data_missing

In [194]:
# Find missing data count per column (commented out after observing)
# elo_ranking_raw.isnull().sum()

### Drop rows where rating is missing using SQL

##### Please refer to the README file for the reasoning of this

#### Clean missing rating column using SQL

In [195]:
# Create a copy of the original DataFrame to prevent changing raw data
elo_ranking1 = elo_ranking_raw.copy()

# Set up in-memory SQL database to allow SQL cleaning
conn = sqlite3.connect(":memory:")

# Load DataFrame into SQLite as a table
elo_ranking1.to_sql("elo_ranking1", conn, if_exists = "replace", index=False)

# Replace "None", "NaN" and NULL values with an empty string
elo_ranking1 = conn.execute("""
    UPDATE elo_ranking1
    SET rating = ""
    WHERE rating IS NULL OR rating = 'None' OR rating = 'NaN'
""")
conn.commit()

# Print the rating, date and team columns to check this worked
rating_updated = pd.read_sql("""
    SELECT rating, date, team
    FROM elo_ranking1
""", conn)

# Print to check this worked (commented out after observing)
# print(rating_updated.to_string(index = False))


#### Check we are Dropping the Correct Rows Using SQL

In [196]:
# Check we are dropping the correct rows
check_empty = pd.read_sql("""
    SELECT *
    FROM elo_ranking1
    WHERE rating = ""
""", conn)

# Print to check (commented out after observing)
#print(check_empty.to_string(index = False))

# Note: all empty ratings are Moldova


#### Delete the Rows Using SQL

In [197]:
# Dropping the missing data
elo_ranking1 = conn.execute("""
    DELETE FROM elo_ranking1
    WHERE rating = ""
""")
conn.commit()

# Check this worked
rows_not_deleted = pd.read_sql("""
    SELECT *
    FROM elo_ranking1
""", conn)

# Print to check (commented out after observing)
# print(rows_not_deleted.to_string(index = False))

#### Check Rows were Deleted Properly

In [198]:
# Convert SQLite elo_ranking1 to pandas DataFrame
elo_ranking1 = pd.read_sql("""
    SELECT * 
    FROM elo_ranking1
""", conn)

# Return how many null in each column (commented out after observing)
# elo_ranking1.isnull().sum()

### Clean date column to be year only in elo_ranking

In [199]:
# Make copy of table to avoid changing raw data
elo_ranking2 = elo_ranking1.copy()

# Clean to year only (from mixed formatting)
elo_ranking2["date"] = pd.to_datetime(
    elo_ranking2["date"], format = "mixed"
).dt.year

# Check this worked for both date formats (commented out after observing)
# print(elo_ranking2)

### Check for duplicate years in each country for elo_ranking

#### Check how many duplicates are in the DataFrame

In [200]:
# Check how many duplicates are in the DataFrame
duplicates_count = elo_ranking2.duplicated(subset = ["date", "team"]).sum()
# Print to check (commented out after observing)
# print(duplicates_count)

### Clean to get rid of duplicates in elo_ranking

#### Create an average for the ratings column where entries have the same year and team

In [201]:
# Group together entries where date (year) and team are the same, and make the rating the average of these previous entries
elo_ranking3 = elo_ranking2.groupby(["date", "team"], as_index = False)["rating"].mean()
# Print to check (commented out after observing)
# print(elo_ranking3)

#### Check this worked by re-checking for duplicates

In [202]:
# Check how many duplicates are in the DataFrame
duplicates_count = elo_ranking3.duplicated(subset = ["date", "team"]).sum()
# Print to check (commented out after observing)
# print(duplicates_count)

## Webscrape the Names of All Participant Countries

### Fetch and Locate Hosts Data

In [203]:
# Fetch HTML from Wikipedia page
participants_url = "https://en.wikipedia.org/wiki/National_team_appearances_in_the_FIFA_World_Cup"

# Create request with custom User Agent to avoid having the request blocked
req = urllib.request.Request(participants_url, headers={'User-Agent' : "Magic Browser"}) 

# Retrieve page content
con = urllib.request.urlopen(req)

# Parse the HTML using BeautifulSoup
soup = BeautifulSoup(con.read(), "html.parser")

# Find all tables with the class "wikitable"
wiki_table_participants = soup.find_all("table", class_=lambda x: x and "wikitable" in x)

# Print number of tables found (commented out after observing)
#print(len(wiki_table_participants))

# Find table with host countries and year
participants_table = None
for table in wiki_table_participants:
    table_header = table.find_all("th") # Get all header cells
    table_header_txt = " ".join([h.text.strip().lower()
                                 for h in table_header]) # Make a list of lowercase header cells so key words can be searched for
    # Print all header strings so we can choose the most relevant table (commented out after observing)
    # print(table_header_txt)
    if "team" in table_header_txt and "appearances" in table_header_txt:
        participants_table = table # Select the table that has "team" and "appearances" as headers
        break

# Print to confirm we have the right wikitable (commented out after observing)        
# print(participants_table)

### Extract Hosts Data to a DataFrame and Export Raw Data to CSV

In [204]:
# Get every row from the table
rows = participants_table.find_all('tr')

data = []

# Skip the first row (column names) and extract the rest
for row in rows[1:]:
    cols = [c.text.strip() for c 
            in row.find_all(['td', 'th'])] # Extract <td> and <th> without spaces
    data.append(cols)

# Convert to DataFrame
participants = pd.DataFrame(data, columns = ["Team", "Appearances", "Record Streak", "Active Streak", "Debut", 
                                   "Most Recent Qualification", "Best Result"])

# Print out to check DataFrame (commented out after observing)
# print(participants)

# Save raw DataFrame as a CSV file
participants.to_csv("participants_raw.csv")


## Cleaning Participant Country DataFrame

### Delete the countries that have not yet debuted (debut in 2026)

In [205]:
# Create a copy of the original DataFrame to prevent changing raw data
participants1 = participants.copy()

# Set up in-memory SQL database to allow SQL cleaning
conn = sqlite3.connect(":memory:")

# Load DataFrame into SQLite as a table
participants1.to_sql("participants1", conn, if_exists="replace", index=False)

# Dropping the missing data
participants1 = conn.execute("""
    DELETE FROM participants1
    WHERE Debut = 2026
""")
conn.commit()

# Check this worked
rows_deleted = pd.read_sql("""
    SELECT *
    FROM participants1
""", conn)

# Print to check (commented out after observing)
# print(rows_deleted.to_string(index=False))


## Webscrape Stats Per Country Per Year

In [206]:
# Extract World Cup  years from Hosts DataFrame
years = host_df1["Year"].tolist()

# Check this worked (commented out after observing)
# print(years)

""" Remove 1978, 1990 and 2002 from the 'years' variable since they do not display the same result when ran 
since they have different column names """
years.remove("1978")
years.remove("1990")
years.remove("2002")

# Check this worked (commented out after observing)
# print(years)

all_rows = []

# Webscrape final rankings by looping through each World Cup year's wikipedia page
for year in years:

    # Fetch HTML from each year's Wikipedia page
    points_url = f"https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup"

    # Print to check the URLs work (commented out after observing)
    # print("URL:",points_url)

    # Create request with custom User Agent to avoid having the request blocked
    req = urllib.request.Request(points_url, headers={'User-Agent' : "Magic Browser"})

    # Retrieve page content
    con = urllib.request.urlopen(req)

    # Parse the HTML using BeautifulSoup
    soup = BeautifulSoup(con.read(), "html.parser")

    # Find all tables with the class "wikitable" on each page individually
    wiki_table_points = soup.find_all("table", class_="wikitable")

    # Identify the "final ranking" table on each page individually
    points_table = None
    for table in wiki_table_points:

        if points_table is not None:
            break
        header = table.find("tr")
        if header:
            first_th = header.find("th")

            # Identify the "final ranking" table by detecting < abbr > title = "final ranking" 
            if first_th:
                abbr = first_th.find("abbr")
                if abbr and "final ranking" == abbr.get("title", "").lower():
                    points_table = table
     
            if points_table is None:
            # Extract rows from the table

                """ If this does not return a table, identify the "final ranking" table based on length 
                (longer than knockout round tables which are all less than 8 historically) and header text """
                if points_table is None and len(table.find_all("tr")) > 8:
                    last_th = header.find_all("th")[-1]
                    if last_th:
                        text = last_th.text.strip()
                    
                    # Print header text to check what is happening (commented out after observing)
                        # print ("TEXT:", text)

                    # If header equals "Result", treat this as the "final ranking" table
                        if text and "Result" == text:
                            points_table = table

    if points_table is not None:
        # Extract rows from the table 
        rows = points_table.find_all("tr")
    
        for row in rows[1:]:  # Skip header row
            cols = row.find_all(["td", "th"]) # Some years have 13 columns - drop column 13
            
            # Print number of columns to check what is happening (commented out after observing)
            # print("COLLEN: ", len(cols))

            # Clean text from each cell
            cols = [col.get_text(strip=True) for col in cols]
    
            if cols:
                all_rows.append([year] + cols[:11])
    
        # Create DataFrame
        total_points_table_raw = pd.DataFrame(all_rows, columns = ["Year", "Ranking", "Team", "Group", "Matches Played", "Matches Won", 
                                           "Matches Drawn", "Matches Lost", "Goals For", "Goals Against",
                                           "Goal Difference", "Points"])
    
        #print(total_points_table)

    # else:
        # print(f"TABLE NOT FOUND{year}")  # Returns if any years haven't worked so code can be edited (commented out after observing)                      

# Print to check this has worked (commented out after observing)
# print(total_points_table_raw.to_string(index=False))

# Save raw DataFrame as a CSV file
total_points_table_raw.to_csv("total_points_table_raw.csv")


In [207]:
# Extract World Cup  years from Hosts DataFrame
years = ["1978", "1990", "2002"]

# Check this worked (commented out after observing)
# print(years)

all_rows = []

# Webscrape final rankings by looping through each World Cup year's wikipedia page
for year in years:

    # Fetch HTML from each year's Wikipedia page
    points_url = f"https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup"

    # Print to check the URLs work (commented out after observing)
    # print("URL:",points_url)

    # Create request with custom User Agent to avoid having the request blocked
    req = urllib.request.Request(points_url, headers={'User-Agent' : "Magic Browser"})

    # Retrieve page content
    con = urllib.request.urlopen(req)

    # Parse the HTML using BeautifulSoup
    soup = BeautifulSoup(con.read(), "html.parser")

    # Find all tables with the class "wikitable" on each page individually
    wiki_table_points = soup.find_all("table", class_="wikitable")

    # Identify the "final ranking" table on each page individually
    points_table = None
    for table in wiki_table_points:

        if points_table is not None:
            break
        header = table.find("tr")
        if header:
            first_th = header.find("th")

            # Identify the "final ranking" table by detecting < abbr > title = "final ranking" 
            if first_th:
                abbr = first_th.find("abbr")
                if abbr and "final ranking" == abbr.get("title", "").lower():
                    points_table = table
     
            if points_table is None:
            # Extract rows from the table

                """ If this does not return a table, identify the "final ranking" table based on length 
                (longer than knockout round tables which are all less than 8 historically) and header text """
                if points_table is None and len(table.find_all("tr")) > 8:
                    last_th = header.find_all("th")[-1]
                    if last_th:
                        text = last_th.text.strip()
                    
                    # Print header text to check what is happening (commented out after observing)
                        # print ("TEXT:", text)

                    # If header equals "Result", treat this as the "final ranking" table
                        if text and "Result" == text:
                            points_table = table

    if points_table is not None:
        # Extract rows from the table 
        rows = points_table.find_all("tr")
    
        for row in rows[1:]:  # Skip header row
            cols = row.find_all(["td", "th"]) # Some years have 13 columns - drop column 13
            
            # Print number of columns to check what is happening (commented out after observing)
            # print("COLLEN: ", len(cols))

            # Clean text from each cell
            cols = [col.get_text(strip=True) for col in cols]
    
            if cols:
                all_rows.append([year] + cols[:12])
    
        # Create DataFrame
        total_points_table_2_raw = pd.DataFrame(all_rows, columns = ["Year", "Ranking", "Group", "Team", "Matches Played", "Matches Won", 
                                           "Matches Drawn", "Matches Lost", "Goals For", "Goals Against",
                                           "Goal Difference", "Points", "Result"])
    

    # else:
        # print(f"TABLE NOT FOUND{year}")  # Returns if any years haven't worked so code can be edited (commented out after observing)                      

# Print to check this has worked (commented out after observing)
# print(total_points_table_2_raw.to_string(index=False))

# Save raw DataFrame as a CSV file
total_points_table_2_raw.to_csv("total_points_table_2_raw.csv")




## Clean and Merge Stats Per Country Per Year DataFrames

### Delete non-matching or irrelevant columns to be prepared for merge

In [208]:
# Create a copy of the original DataFrame to prevent changing raw data
total_points_table_1_1 = total_points_table_raw.copy()
total_points_table_2_1 = total_points_table_2_raw.copy()

# Delete "Result" and "Group" columns from the 2nd stats per country per year df
total_points_table_2_1 = total_points_table_2_1.drop(columns=["Result"])
total_points_table_2_1 = total_points_table_2_1.drop(columns=["Group"])

# Print to check this worked (commented out after observing)
# print(total_points_table_2_1.to_string(index = False))


In [209]:
# Delete "Group" column from the 1st stats per country per year df
total_points_table_1_1 = total_points_table_1_1.drop(columns=["Group"])

# Print to check this worked (commented out after observing)
#print(total_points_table_1_1.to_string(index = False))

### Concatenate the DataFrames

In [210]:
# Concatenate total_points_table_1_1 and total_points_table_2_1
df_combined = pd.concat([total_points_table_1_1, total_points_table_2_1], ignore_index=True)

# Print to check this worked (commented out after observing)
# print(df_combined.to_string(index = False))

### Get rid of NULL entries (e.g. subheadings that were scraped)

#### Check for NULL entries

In [238]:
# Print DataFrame to find NULL entries (commented out after observing)
# df_combined.isnull()

In [243]:
# Sum for NULL entries in columns (commented out after observing)
# df_combined.isnull().sum()

#### Make all NULL entries have an identifiable characteristic using SQL
##### Please refer to the README for reasoning as to why "Points" was chosen as this identifiable characteristic

In [244]:
# Create a copy of the original DataFrame to prevent changing raw data
df_combined1 = df_combined.copy()

# Set up in-memory SQL database to allow SQL cleaning
conn = sqlite3.connect(":memory:")

# Load DataFrame into SQLite as a table
df_combined1.to_sql("df_combined1", conn, if_exists = "replace", index=False)

# Replace "None", "NaN" and NULL values with an empty string
df_combined1 = conn.execute("""
    UPDATE df_combined1
    SET Points = ""
    WHERE Points IS NULL OR Points = 'None' OR Points = 'NaN'
""")
conn.commit()

# Print the DataFrame to check this worked
points_updated = pd.read_sql("""
    SELECT *
    FROM df_combined1
""", conn)

# Print to check this worked (commented out after observing)
# print(points_updated.to_string(index = False))

# Print to check length of DataFrame (commented out after observing)
# print(len(points_updated))

#### Drop NULL entries

In [245]:
# Dropping the missing data
delete_rows = conn.execute("""
    DELETE FROM df_combined1
    WHERE Points = ""
""")
conn.commit()

# Check this worked
df_combined2 = pd.read_sql("""
    SELECT *
    FROM df_combined1
""", conn)

# Print to check (commented out after observing)
# print(df_combined2.to_string(index = False))

# Print to check length of DataFrame (commented out after observing)
# print(len(df_combined2))

#### Check for NULL entries

In [246]:
# Print DataFrame to find NULL entries (commented out after observing)
# df_combined2.isnull()

In [247]:
# Sum for NULL entries in columns (commented out after observing)
# df_combined2.isnull().sum()